# FGAI (ICT4403) Assignment 3 - EN→HI MT on **Google Colab**
**Before running:** Runtime → Change runtime type → **T4 GPU**. Left sidebar → 🔑 Secrets → add `HF_TOKEN` (your Hugging Face read token) and switch on *Notebook access*.
Run the cells top to bottom. First do a tiny test (`TINY_RUN = True`), then set it to `False` for the real run.
If Colab disconnects, reconnect and run all cells again - finished stages are skipped (results are kept on your Google Drive).

## 1. Settings

In [ ]:
TINY_RUN = True           # True = tiny pipeline check (minutes). False = the real experiment.
CLEAN_CHECKPOINTS = True  # delete big model weights at the end (frees Google Drive space)
REUPLOAD = False          # True = delete the old copy of the code on Drive and upload the zip again (use after getting updated code)

# "Light" preset = smaller training subsets (Llama-3.2-1B is already the smallest Llama 3.x).
# Set LIGHT = False for the full-size experiment (Gemma-3 1B, more training data).
LIGHT = True
LIGHT_SETTINGS = {
    # "GEMMA_ID": "google/gemma-3-270m",   # optional even smaller Gemma (accept its licence first)
    "N_TRAIN_SCRATCH": "50000", "SCRATCH_EPOCHS": "10",
    "N_TRAIN_MT5": "20000",     "MT5_EPOCHS": "2",
    "N_TRAIN_LLM": "5000",
    "N_TEMP_TEST": "100",
}
OVERRIDES = dict(LIGHT_SETTINGS) if LIGHT else {}
# Add your own overrides here if a stage is still too slow, e.g.  OVERRIDES["N_TRAIN_MT5"] = "10000"

TINY = dict(POOL_SIZE="2000", N_TEST="50", N_TEMP_TEST="20", N_TRAIN_SCRATCH="2000", N_TRAIN_MT5="1000",
            N_TRAIN_LLM="500", SCRATCH_EPOCHS="2", MT5_EPOCHS="1", SCRATCH_VOCAB="4000")

## 2. Get the code (upload `en-hi-mt-comparison.zip` once) and check the GPU

In [ ]:
import os, shutil, subprocess, sys, zipfile, torch
from google.colab import drive, files

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")                       # results/checkpoints live here so they survive disconnects
BASE = "/content/drive/MyDrive"
REPO = f"{BASE}/en-hi-mt-comparison"
if REUPLOAD and os.path.exists(REPO):
    shutil.rmtree(REPO)                             # remove the old code (results inside it are removed too)
if not os.path.exists(f"{REPO}/src/config.py"):
    print("Upload en-hi-mt-comparison.zip now:")
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(BASE)
assert os.path.exists(f"{REPO}/src/config.py"), "Code not found - upload the zip that contains the en-hi-mt-comparison folder."

if TINY_RUN:                                        # the tiny test gets its own throw-away copy so it never mixes with real results
    WORK = "/content/en-hi-mt-tiny"
    shutil.rmtree(WORK, ignore_errors=True)
    shutil.copytree(REPO, WORK, ignore=shutil.ignore_patterns("checkpoints", "data", "results"))
    for d in ["data", "results", "checkpoints"]:
        os.makedirs(f"{WORK}/{d}", exist_ok=True)
else:
    WORK = REPO
os.chdir(WORK)
print("working in:", WORK, "| existing results:", sorted(os.listdir("results")) if os.path.exists("results") else [])

## 3. Install dependencies

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
assert r.returncode == 0, "pip install failed"
# peft + the old torchao that Kaggle ships are incompatible (LoRA stages crash) - torchao is not needed, remove it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True)
import importlib.metadata as md_
for p in ["torch", "transformers", "peft", "datasets", "sacrebleu", "sentencepiece"]:
    print(p, md_.version(p))

## 4. Hugging Face token + access check (fails fast instead of after hours)

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token (hf_...): ")

from huggingface_hub import hf_hub_download
hf_hub_download("cfilt/iitb-english-hindi", "README.md", repo_type="dataset"); print("OK dataset reachable")
for repo_id in [OVERRIDES.get("LLAMA_ID", "meta-llama/Llama-3.2-1B-Instruct"), OVERRIDES.get("GEMMA_ID", "google/gemma-3-1b-it")]:
    try:
        hf_hub_download(repo_id, "config.json"); print("OK access:", repo_id)
    except Exception as e:
        raise RuntimeError(f"No access to {repo_id}. Accept its licence on huggingface.co with the SAME account as the token. ({type(e).__name__})")

## 5. Helpers

In [ ]:
os.environ.update(OVERRIDES)
if TINY_RUN:
    os.environ.update(TINY)          # tiny sizes win over OVERRIDES (but GEMMA_ID stays)
FAILED = []
def run(cmd, required=True):
    """Run a shell command and STREAM its output into the cell (subprocess output is otherwise invisible in notebooks)."""
    print("\n>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        FAILED.append(cmd)
        if required:
            raise RuntimeError(f"FAILED: {cmd}")
        print(f"FAILED: {cmd} (continuing)")

def stage(name, cmd, required=True):
    """Skip a stage whose result file already exists, so re-running after a disconnect resumes where it stopped."""
    if os.path.exists(f"results/{name}.json"):
        print(f"SKIP {name}: results/{name}.json already exists")
        return
    run(cmd, required)

## 6. Data

In [ ]:
if os.path.exists("data/test.jsonl"):
    print("SKIP data: already prepared")
else:
    run("python -m src.data")

## 7. Model 1 - Basic Transformer (from scratch)

In [ ]:
stage('transformer', 'python -m src.train_transformer')

## 8. Model 2 - mT5 fine-tuning

In [ ]:
stage('mt5', 'python -m src.finetune_mt5')

## 9. Model 3 - LLaMA (LoRA)

In [ ]:
stage('llama_lora', 'python -m src.llm_mt --family llama --mode lora')

## 10. Model 4 - Gemma (LoRA)

In [ ]:
stage('gemma_lora', 'python -m src.llm_mt --family gemma --mode lora', required=False)

## 11. Prompting baselines (optional)

In [ ]:
for fam in ["llama", "gemma"]:
    for mode in ["zeroshot", "fewshot"]:
        stage(f"{fam}_{mode}", f"python -m src.llm_mt --family {fam} --mode {mode} --skip_temp", required=False)

## 12. Tokenizer stats, tables, plots, README

In [ ]:
run('python -m src.tokenizer_stats', required=False)
run('python -m src.compare')
print("\nFailed commands:", FAILED or "none")

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/results_table.md").read()))
for f in ["quality_comparison", "compute_comparison", "temperature_analysis", "training_curves"]:
    if os.path.exists(f"results/{f}.png"):
        display(Image(f"results/{f}.png"))

## 13. Bundle and download the results

In [ ]:
with zipfile.ZipFile("results_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk("results"):
        for f in fs:
            z.write(os.path.join(root, f))
    z.write("README.md")
if CLEAN_CHECKPOINTS and not TINY_RUN:
    shutil.rmtree("checkpoints", ignore_errors=True)
files.download("results_bundle.zip")